# Лабораторная работа №3
## Современные архитектуры для последовательностей: LSTM vs TCN vs Transformer vs Mamba

Цель: освоить современные архитектуры для работы с последовательностями, понять разницу между рекуррентными, сверточными, трансформерными и State Space моделями, научиться работать с аудио-данными через спектрограммы.

In [ ]:
# === ПРОВЕРКА СРЕДЫ ВЫПОЛНЕНИЯ ===
import torch
if torch.cuda.is_available():
    print(f"GPU доступен: {torch.cuda.get_device_name(0)}")
else:
    print("GPU не найден, будет использован CPU (обучение займет больше времени)")

In [ ]:
# === ИМПОРТЫ ===
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, roc_auc_score)
from sklearn.model_selection import train_test_split
import time

import copy, math, random
import pandas as pd
import torch.nn.functional as F
import matplotlib.ticker as mticker
from IPython.display import display, Markdown

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def sync():
    if device.type == 'cuda':
        torch.cuda.synchronize()


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3, 'lines.linewidth': 2})
COLORS = {'LSTM': '#2a78d6', 'TCN': '#eb6834', 'PatchTST': '#1baf7a', 'Mamba': '#eda100'}

In [ ]:
# ============================================================
# ЧАСТЬ 1. Классификация ЭКГ: сравнение архитектур
# ============================================================
# ЗАДАНИЕ:
# Использовать heartbeat-датасет (5 классов аритмий).
# Обучить и сравнить 4 архитектуры:
# 1. LSTM (baseline)
# 2. TCN (Temporal Convolutional Network)
# 3. PatchTST (упрощенный Transformer)
# 4. Mamba (State Space Model)
# Измерить время обучения и потребление памяти.
# ============================================================

# === РЕШЕНИЕ ===

# --- 1. Генерация синтетического ЭКГ-датасета ---
SEQ_LEN, N_PER_CLASS, N_CLASSES = 187, 600, 5
ECG_CLASSES = ['N норма', 'S наджелуд.', 'V желудочк.', 'F сливной', 'Q стимулятор']


def gauss(t, mu, sigma, amp):
    return amp * np.exp(-0.5 * ((t - mu) / sigma) ** 2)


def synth_beat(cls, rng):
    t = np.arange(SEQ_LEN, dtype=np.float64)
    k = rng.uniform(0.85, 1.15)
    r = 70 + rng.normal(0, 4)
    a = lambda: rng.uniform(0.8, 1.2)

    def normal():
        return (gauss(t, r - 28 * k, 4 * k, 0.15 * a()) + gauss(t, r - 6, 2, -0.1 * a())
                + gauss(t, r, 2.5, a()) + gauss(t, r + 6, 2.5, -0.2 * a()) + gauss(t, r + 45 * k, 9 * k, 0.3 * a()))

    def ventricular():
        w = rng.uniform(6, 9)
        return gauss(t, r, w, 1.1 * a()) + gauss(t, r + 8, w, -0.3 * a()) + gauss(t, r + 50 * k, 12 * k, -0.35 * a())

    if cls == 0:
        x = normal()
    elif cls == 1:
        r = r - rng.uniform(8, 20)
        x = (gauss(t, r - rng.uniform(12, 22) * k, 4, rng.choice([-1, 1]) * 0.12 * a()) + gauss(t, r - 6, 2, -0.1 * a())
             + gauss(t, r, 2.5, a()) + gauss(t, r + 6, 2.5, -0.2 * a()) + gauss(t, r + 45 * k, 9 * k, 0.3 * a())
             + gauss(t, 5, 10, 0.25 * a()))
    elif cls == 2:
        x = ventricular()
    elif cls == 3:
        alpha = rng.uniform(0.3, 0.7)
        x = alpha * normal() + (1 - alpha) * ventricular()
    else:
        x = (gauss(t, r - 9, 0.7, rng.uniform(0.6, 1.0)) + gauss(t, r, rng.uniform(5, 7), 0.9 * a())
             + gauss(t, r + 50 * k, 11 * k, -0.25 * a()))
    x = x + 0.05 * np.sin(2 * np.pi * rng.uniform(0.002, 0.01) * t + rng.uniform(0, 2 * np.pi))
    x = x + rng.normal(0, 0.03, SEQ_LEN)
    return ((x - x.min()) / (x.max() - x.min())).astype(np.float32)


rng = np.random.default_rng(SEED)
X_ecg = np.stack([synth_beat(c, rng) for c in range(N_CLASSES) for _ in range(N_PER_CLASS)])
y_ecg = np.repeat(np.arange(N_CLASSES), N_PER_CLASS)

X_tmp, X_te, y_tmp, y_te = train_test_split(X_ecg, y_ecg, test_size=450, stratify=y_ecg, random_state=SEED)
X_tr, X_va, y_tr, y_va = train_test_split(X_tmp, y_tmp, test_size=450, stratify=y_tmp, random_state=SEED)
print(f"train {X_tr.shape}, val {X_va.shape}, test {X_te.shape}, по классам в train: {np.bincount(y_tr)}")

fig, axes = plt.subplots(1, N_CLASSES, figsize=(17, 2.8), sharey=True)
for c, ax in enumerate(axes):
    for x in X_ecg[y_ecg == c][:4]:
        ax.plot(x, lw=1.2, alpha=0.8)
    ax.set_title(ECG_CLASSES[c])
    ax.set_xlabel('отсчет')
fig.suptitle('Синтетические циклы ЭКГ, по 4 примера на класс', y=1.04)
plt.show()

In [ ]:
# --- 2. Определение архитектур ---

class LSTMClassifier(nn.Module):
    def __init__(self, hidden=64, layers=2):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden, layers, batch_first=True, bidirectional=True, dropout=0.1)
        self.head = nn.Linear(2 * hidden, N_CLASSES)

    def forward(self, x):
        _, (h, _) = self.lstm(x.unsqueeze(-1))
        return self.head(torch.cat([h[-2], h[-1]], dim=1))


class CausalConv1d(nn.Conv1d):
    def __init__(self, c_in, c_out, k, dilation):
        super().__init__(c_in, c_out, k, dilation=dilation)
        self.left_pad = (k - 1) * dilation

    def forward(self, x):
        return super().forward(F.pad(x, (self.left_pad, 0)))


class TCNBlock(nn.Module):
    def __init__(self, c_in, c_out, k, dilation, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(CausalConv1d(c_in, c_out, k, dilation), nn.ReLU(), nn.Dropout(dropout),
                                 CausalConv1d(c_out, c_out, k, dilation), nn.ReLU(), nn.Dropout(dropout))
        self.res = nn.Conv1d(c_in, c_out, 1) if c_in != c_out else nn.Identity()

    def forward(self, x):
        return F.relu(self.net(x) + self.res(x))


class TCN(nn.Module):
    def __init__(self, channels=32, k=7, dilations=(1, 4, 16)):
        super().__init__()
        self.blocks = nn.Sequential(*[TCNBlock(1 if i == 0 else channels, channels, k, d) for i, d in enumerate(dilations)])
        self.head = nn.Linear(channels, N_CLASSES)

    def forward(self, x):
        return self.head(self.blocks(x.unsqueeze(1))[:, :, -1])


class PatchTST(nn.Module):
    def __init__(self, patch_len=16, stride=8, d=64, depth=2, heads=4):
        super().__init__()
        self.patch_len, self.stride = patch_len, stride
        n_patches = (SEQ_LEN + stride - patch_len) // stride + 1
        self.embed = nn.Linear(patch_len, d)
        self.pos = nn.Parameter(0.02 * torch.randn(1, n_patches, d))
        layer = nn.TransformerEncoderLayer(d, heads, 2 * d, dropout=0.1, activation='gelu', batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Flatten(), nn.Dropout(0.1), nn.Linear(n_patches * d, N_CLASSES))

    def forward(self, x):
        x = F.pad(x.unsqueeze(1), (0, self.stride), mode='replicate').squeeze(1)
        patches = x.unfold(-1, self.patch_len, self.stride)
        return self.head(self.encoder(self.embed(patches) + self.pos))


class SelectiveSSM(nn.Module):
    def __init__(self, d_model, d_state=16, expand=2, d_conv=4):
        super().__init__()
        d_inner, self.d_state = expand * d_model, d_state
        self.dt_rank = math.ceil(d_model / 16)
        self.in_proj = nn.Linear(d_model, 2 * d_inner)
        self.conv = nn.Conv1d(d_inner, d_inner, d_conv, groups=d_inner, padding=d_conv - 1)
        self.x_proj = nn.Linear(d_inner, self.dt_rank + 2 * d_state, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, d_inner)
        self.A_log = nn.Parameter(torch.log(torch.arange(1, d_state + 1).float().repeat(d_inner, 1)))
        self.D = nn.Parameter(torch.ones(d_inner))
        self.out_proj = nn.Linear(d_inner, d_model)
        dt = torch.exp(torch.rand(d_inner) * (math.log(0.1) - math.log(1e-3)) + math.log(1e-3))
        with torch.no_grad():
            self.dt_proj.bias.copy_(dt + torch.log(-torch.expm1(-dt)))

    def forward(self, x):
        L = x.shape[1]
        x, z = self.in_proj(x).chunk(2, dim=-1)
        x = F.silu(self.conv(x.transpose(1, 2))[..., :L].transpose(1, 2))
        dt, Bm, Cm = self.x_proj(x).split([self.dt_rank, self.d_state, self.d_state], dim=-1)
        dt = F.softplus(self.dt_proj(dt))
        A = -torch.exp(self.A_log)
        dA = torch.exp(dt.unsqueeze(-1) * A)
        dBx = dt.unsqueeze(-1) * Bm.unsqueeze(2) * x.unsqueeze(-1)
        h = torch.zeros_like(dA[:, 0])
        ys = []
        for dA_t, dBx_t, C_t in zip(dA.unbind(1), dBx.unbind(1), Cm.unbind(1)):
            h = dA_t * h + dBx_t
            ys.append((h * C_t.unsqueeze(1)).sum(-1))
        y = torch.stack(ys, dim=1) + x * self.D
        return self.out_proj(y * F.silu(z))


class Mamba(nn.Module):
    def __init__(self, d_model=32, depth=2):
        super().__init__()
        self.embed = nn.Linear(1, d_model)
        self.layers = nn.ModuleList([SelectiveSSM(d_model) for _ in range(depth)])
        self.norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(depth)])
        self.norm_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, N_CLASSES)

    def forward(self, x):
        h = self.embed(x.unsqueeze(-1))
        for norm, layer in zip(self.norms, self.layers):
            h = h + layer(norm(h))
        return self.head(self.norm_f(h).mean(dim=1))


MODELS = {'LSTM': LSTMClassifier, 'TCN': TCN, 'PatchTST': PatchTST, 'Mamba': Mamba}
for name, cls in MODELS.items():
    print(f"{name:9s}: {count_params(cls()):6d} параметров")

In [ ]:
# --- 3. Обучение и сравнение ---
EPOCHS, BATCH, LR = 30, 64, 1e-3
to_t = lambda a, dt=torch.float32: torch.as_tensor(a, dtype=dt, device=device)
Xtr_t, ytr_t, Xva_t, yva_t, Xte_t, yte_t = (to_t(X_tr), to_t(y_tr, torch.long), to_t(X_va), to_t(y_va, torch.long),
                                            to_t(X_te), to_t(y_te, torch.long))


@torch.no_grad()
def predict_proba(model, X, bs=512):
    model.eval()
    return torch.cat([model(X[i:i + bs]).softmax(-1) for i in range(0, len(X), bs)]).cpu().numpy()


def train(model, Xtr, ytr, Xva, yva, epochs, bs=BATCH, lr=LR, augment=None):
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    gen = torch.Generator().manual_seed(SEED)
    hist, best = {'train_loss': [], 'val_acc': []}, (-1, None)
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    sync()
    t0 = time.perf_counter()
    for _ in range(epochs):
        model.train()
        perm, total = torch.randperm(len(Xtr), generator=gen).to(device), 0.0
        for i in range(0, len(Xtr), bs):
            idx = perm[i:i + bs]
            xb = Xtr[idx] if augment is None else augment(Xtr[idx])
            loss = F.cross_entropy(model(xb), ytr[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * len(idx)
        sched.step()
        va = float((predict_proba(model, Xva).argmax(1) == yva.cpu().numpy()).mean())
        hist['train_loss'].append(total / len(Xtr))
        hist['val_acc'].append(va)
        if va > best[0]:
            best = (va, copy.deepcopy(model.state_dict()))
    sync()
    hist['time'] = time.perf_counter() - t0
    hist['peak_mem_mb'] = torch.cuda.max_memory_allocated() / 2 ** 20 if device.type == 'cuda' else float('nan')
    model.load_state_dict(best[1])
    return model, hist


ecg_models, ecg_hist, rows = {}, {}, []
for name, cls in MODELS.items():
    set_seed(SEED)
    model, hist = train(cls().to(device), Xtr_t, ytr_t, Xva_t, yva_t, EPOCHS)
    ecg_models[name], ecg_hist[name] = model, hist
    sync()
    t0 = time.perf_counter()
    proba = predict_proba(model, Xte_t)
    sync()
    infer_ms = 1000 * (time.perf_counter() - t0)
    pred = proba.argmax(1)
    rows.append({'Модель': name, 'Параметров': count_params(model), 'Время 30 эпох, с': hist['time'],
                 'Best val acc': max(hist['val_acc']), 'Test acc': (pred == y_te).mean(),
                 'Test macro-F1': f1_score(y_te, pred, average='macro'),
                 'Test ROC-AUC': roc_auc_score(y_te, proba, multi_class='ovr'),
                 'Инференс теста (450), мс': infer_ms, 'Пик памяти GPU, МБ': hist['peak_mem_mb']})
    print(f"{name:9s} | {hist['time']:6.1f} c | best val acc {max(hist['val_acc']):.3f} | test acc {rows[-1]['Test acc']:.3f}")

ecg_table = pd.DataFrame(rows).set_index('Модель')
display(ecg_table.round(3))

In [ ]:
# --- 4. Визуализация ---
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
ep = np.arange(1, EPOCHS + 1)
for name, h in ecg_hist.items():
    axes[0].plot(ep, h['train_loss'], color=COLORS[name], label=name)
    axes[1].plot(ep, h['val_acc'], color=COLORS[name], label=name)
    axes[2].scatter(h['time'], ecg_table.loc[name, 'Test acc'], s=90, color=COLORS[name], zorder=3)
    axes[2].annotate(name, (h['time'], ecg_table.loc[name, 'Test acc']), xytext=(6, 4), textcoords='offset points')
axes[0].set_yscale('log')
axes[0].set_title('Train loss')
axes[0].set_xlabel('эпоха')
axes[0].legend()
axes[1].set_title('Val accuracy')
axes[1].set_xlabel('эпоха')
axes[1].legend()
axes[2].set_xscale('log')
axes[2].xaxis.set_major_formatter(mticker.ScalarFormatter())
axes[2].set_xlabel('время обучения 30 эпох, с')
axes[2].set_ylabel('test accuracy')
axes[2].set_title('Скорость и качество')
plt.tight_layout()
plt.show()

conf_ecg = {}
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, m) in zip(axes, ecg_models.items()):
    cm = confusion_matrix(y_te, predict_proba(m, Xte_t).argmax(1))
    conf_ecg[name] = cm
    ax.imshow(cm, cmap='Blues')
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=8, color='white' if cm[i, j] > 50 else 'black')
    ax.set_xticks(range(N_CLASSES), [c.split()[0] for c in ECG_CLASSES])
    ax.set_yticks(range(N_CLASSES), [c.split()[0] for c in ECG_CLASSES])
    ax.set_title(name)
    ax.set_xlabel('прогноз')
    ax.grid(False)
axes[0].set_ylabel('истина')
plt.tight_layout()
plt.show()

### Выводы по части 1

In [ ]:
t = ecg_table
names = list(t.index)
best = t['Test acc'].idxmax()
by_time = t['Время 30 эпох, с'].sort_values()
smallest = t['Параметров'].idxmin()
code = [c.split()[0] for c in ECG_CLASSES]

total_cm = sum(conf_ecg.values())
off = total_cm - np.diag(np.diag(total_cm))
sym = np.triu(off + off.T, k=1)
pa, pb = divmod(int(sym.argmax()), N_CLASSES)
pair_note = {
    frozenset({2, 4}): "у обоих классов широкий QRS без зубца P, отличает их только узкий спайк стимулятора перед комплексом",
    frozenset({2, 3}): "сливной комплекс - смесь нормального и желудочкового, поэтому он похож на V",
    frozenset({3, 4}): "у обоих классов расширенный QRS, а спайк стимулятора короткий и легко теряется",
    frozenset({0, 3}): "сливной комплекс частично повторяет нормальный",
    frozenset({0, 1}): "наджелудочковый комплекс по форме почти нормальный, отличается в основном временем прихода",
}

text = ["- Test accuracy: " + ', '.join(f"{n} - {t.loc[n, 'Test acc']:.3f}" for n in names)
        + f". Лучшая модель - {best} (macro-F1 {t.loc[best, 'Test macro-F1']:.3f})."]
s = f"- Чаще всего модели путают классы {code[pa]} и {code[pb]} (ошибок по всем моделям: {sym[pa, pb]})"
s += (f": {pair_note[frozenset({pa, pb})]}." if frozenset({pa, pb}) in pair_note else ".")
text.append(s)
text.append(f"- Время обучения ({device.type.upper()}, {EPOCHS} эпох): "
            + ', '.join(f"{n} - {v:.0f} с" for n, v in by_time.items())
            + f". Быстрее всего {by_time.index[0]}, медленнее всего {by_time.index[-1]}.")
text.append("- PatchTST режет сигнал на 23 патча и считает attention по ним сразу, TCN считает свертки параллельно "
            "по всей длине, а LSTM проходит 187 шагов последовательно. В нашем Mamba скан по времени тоже написан "
            "циклом, поэтому он медленный; в настоящем Mamba этот скан параллельный и работает быстро на GPU.")
text.append(f"- Меньше всего параметров у {smallest} ({t.loc[smallest, 'Параметров']}), больше всего у "
            f"{t['Параметров'].idxmax()} ({t['Параметров'].max()}). Число параметров не определяет качество.")
jumps = int((np.diff(ecg_hist['LSTM']['val_acc']) < -0.1).sum())
if jumps:
    text.append(f"- У LSTM val accuracy {jumps} раз резко падала между эпохами, обучение менее стабильное.")
why_fast = {'PatchTST': "патчи сокращают длину последовательности, и attention сразу видит весь цикл",
            'TCN': "свертки считаются параллельно, а рецептивное поле покрывает весь цикл"}
if best in list(by_time.index[:2]):
    text.append(f"- Trade-off: {best} оказался и самым точным, и одним из самых быстрых"
                + (f": {why_fast[best]}." if best in why_fast else "."))
else:
    text.append(f"- Trade-off: лучшая точность у {best}, но быстрее обучается {by_time.index[0]} "
                f"(accuracy {t.loc[by_time.index[0], 'Test acc']:.3f}). Выбор зависит от того, что важнее.")
display(Markdown('\n'.join(text)))

In [ ]:
# ============================================================
# ЧАСТЬ 2. Робастность к шуму
# ============================================================
# ЗАДАНИЕ:
# Добавить шум к тестовым сигналам и измерить падение accuracy.
# ============================================================

# === РЕШЕНИЕ ===
SIGMAS, N_REP = [0.0, 0.1, 0.3, 0.5, 0.8, 1.0], 10
noise_rng = np.random.default_rng(SEED)
noise_acc = {name: np.zeros((len(SIGMAS), N_REP)) for name in ecg_models}
for i, s in enumerate(SIGMAS):
    for r in range(N_REP):
        Xn = to_t(X_te + noise_rng.normal(0, s, X_te.shape).astype(np.float32))
        for name, m in ecg_models.items():
            noise_acc[name][i, r] = (predict_proba(m, Xn).argmax(1) == y_te).mean()

fig, axes = plt.subplots(1, 2, figsize=(15, 4.3), gridspec_kw={'width_ratios': [1.3, 1]})
for name, acc in noise_acc.items():
    axes[0].errorbar(SIGMAS, acc.mean(1), yerr=acc.std(1), fmt='o-', color=COLORS[name], capsize=3, label=name)
axes[0].axhline(1 / N_CLASSES, color='0.6', ls=':', lw=1, label='случайное угадывание')
axes[0].set_xlabel('sigma шума')
axes[0].set_ylabel('test accuracy')
axes[0].set_title(f'Робастность к шуму (среднее и std по {N_REP} повторам)')
axes[0].legend()
for s, shift in zip([0.0, 0.1, 0.3, 1.0], [0, 1.2, 2.4, 3.6]):
    axes[1].plot(X_te[0] + np.random.default_rng(1).normal(0, s, SEQ_LEN) * (s > 0) + shift, lw=1, label=f'sigma = {s}')
axes[1].set_title(f'Один цикл ({ECG_CLASSES[y_te[0]]}) при разном шуме')
axes[1].set_yticks([])
axes[1].legend(loc='upper left', bbox_to_anchor=(1.0, 1.0), fontsize=8)
plt.tight_layout()
plt.show()

noise_df = pd.DataFrame({n: a.mean(1) for n, a in noise_acc.items()}, index=pd.Index(SIGMAS, name='sigma'))
noise_df.loc['Средняя acc при sigma > 0'] = noise_df.iloc[1:].mean()
display(noise_df.round(3))

### Выводы по части 2

In [ ]:
mean_noisy = noise_df.loc['Средняя acc при sigma > 0']
rank = mean_noisy.sort_values(ascending=False)
i01, i03, i10 = SIGMAS.index(0.1), SIGMAS.index(0.3), SIGMAS.index(1.0)
drop01 = {n: (noise_acc[n][0].mean() - noise_acc[n][i01].mean()) * 100 for n in ecg_models}

text = ["- Accuracy при sigma = 0.3: " + ', '.join(f"{n} - {noise_acc[n][i03].mean():.3f}" for n in ecg_models)
        + "; при sigma = 1.0: " + ', '.join(f"{n} - {noise_acc[n][i10].mean():.3f}" for n in ecg_models) + ".",
        f"- По средней accuracy при sigma > 0 модели располагаются так: "
        + ', '.join(f"{n} ({v:.3f})" for n, v in rank.items())
        + f". Самая устойчивая - {rank.index[0]}, самая чувствительная - {rank.index[-1]}.",
        "- Изменение accuracy уже при sigma = 0.1: " + ', '.join(f"{n} {-v:+.1f} п.п." for n, v in drop01.items())]

for other in ['TCN', 'Mamba']:
    if mean_noisy[other] > mean_noisy['LSTM'] + 0.01:
        text.append(f"- {other} устойчивее LSTM (средняя accuracy {mean_noisy[other]:.3f} против {mean_noisy['LSTM']:.3f}), "
                    "это совпадает с ожиданием.")
    elif mean_noisy[other] < mean_noisy['LSTM'] - 0.01:
        text.append(f"- {other} в этом эксперименте менее устойчив, чем LSTM ({mean_noisy[other]:.3f} против "
                    f"{mean_noisy['LSTM']:.3f}), ожидание не подтвердилось.")
    else:
        text.append(f"- {other} и LSTM под шумом примерно одинаковы ({mean_noisy[other]:.3f} и {mean_noisy['LSTM']:.3f}).")

text.append("- Почему TCN и Mamba обычно устойчивее LSTM. Свертка TCN суммирует сигнал по окну, то есть работает как "
            "обучаемый фильтр: независимый шум в соседних точках частично усредняется, а форма зубцов сохраняется. "
            "SSM в Mamba тоже фильтр с памятью, он сглаживает вход по времени. LSTM читает сигнал по одной точке, и шум "
            "каждого шага проходит через гейты и копится в скрытом состоянии.")
text.append("- У Mamba есть особенность: dt, B и C считаются из каждого отсчета входа, поэтому шум меняет не только "
            "сигнал, но и сам фильтр (что запомнить, что забыть). Маленькая модель, обученная на чистых данных, "
            "может быть к этому чувствительна.")
text.append("- Природа признаков: TCN опирается на локальную форму комплекса, сглаженную по окну; PatchTST - на точную "
            "форму патчей и их взаимное положение, поэтому шум сразу попадает в токены; LSTM и Mamba - на "
            "последовательную динамику отсчетов. Ни одна модель не видела шума при обучении, обычный способ повысить "
            "устойчивость - добавлять шум как аугментацию.")
display(Markdown('\n'.join(text)))

In [ ]:
# ============================================================
# ЧАСТЬ 3. Классификация аудио (ESC-50 / Speech Commands)
# ============================================================
# ЗАДАНИЕ:
# Классифицировать звуки по спектрограммам с помощью
# Audio Spectrogram Transformer (AST) - ViT для аудио.
# ============================================================

# === РЕШЕНИЕ ===

# --- 3.1 Синтетические лог-спектрограммы 64x128 (частота x время), 10 классов по 200 ---
F_BINS, T_FRAMES, N_SOUND = 64, 128, 200
SOUNDS = ['сирена', 'лай собаки', 'тиканье часов', 'дождь', 'двигатель',
          'птицы', 'речь', 'стук в дверь', 'свист', 'гром']


def make_spectrogram(cls, rng):
    f = np.arange(F_BINS, dtype=float)[:, None]
    t = np.arange(T_FRAMES, dtype=float)[None, :]
    u = rng.uniform
    line = lambda f0, w=1.2: np.exp(-0.5 * ((f - f0) / w) ** 2)
    burst = lambda t0, tau: np.exp(-np.clip(t - t0, 0, None) / tau) * (t >= t0)
    S = np.zeros((F_BINS, T_FRAMES))
    if cls == 0:
        f0 = u(22, 38) + u(5, 10) * np.sin(2 * np.pi * t / u(30, 70) + u(0, 6.3))
        S += line(f0) + 0.4 * line(f0 + 14)
    elif cls == 1:
        for _ in range(rng.integers(2, 5)):
            e, f0 = burst(u(0, T_FRAMES - 12), u(3, 6)), u(6, 12)
            S += e * (sum(line(h * f0, 1.5) / h ** 0.5 for h in range(1, 6)) + 0.3)
    elif cls == 2:
        per = u(8, 14)
        S += np.exp(-0.5 * (((t - u(0, per)) % per) / 0.6) ** 2) * (f > u(15, 30))
    elif cls == 3:
        S += u(0.3, 0.6) * (0.5 + f / F_BINS) * rng.random((F_BINS, T_FRAMES))
        S[rng.integers(10, F_BINS, 80), rng.integers(0, T_FRAMES, 80)] += u(0.5, 1.0)
    elif cls == 4:
        f0, mod = u(2.5, 5), 1 + 0.2 * np.sin(2 * np.pi * t / u(20, 60))
        S += mod * sum(line(h * f0, 0.8) / h for h in range(1, 9) if h * f0 < 40)
    elif cls == 5:
        for _ in range(rng.integers(3, 7)):
            t0, dur, f_start, slope = u(0, T_FRAMES - 8), u(3, 7), u(38, 52), u(1, 2.5)
            S += line(f_start + slope * (t - t0), 1.0) * ((t >= t0) & (t < t0 + dur))
    elif cls == 6:
        env, t0 = np.zeros_like(t), u(0, 10)
        for _ in range(rng.integers(3, 6)):
            dur = u(10, 18)
            env += (t >= t0) & (t < t0 + dur)
            t0 += dur + u(4, 12)
        f0 = u(5, 8) + 1.5 * np.sin(2 * np.pi * t / u(40, 90) + u(0, 6.3))
        formants = np.exp(-((f - u(15, 25)) / 6) ** 2) + 0.7 * np.exp(-((f - u(30, 42)) / 6) ** 2)
        S += env * formants * sum(line(h * f0, 1.0) for h in range(1, 9))
    elif cls == 7:
        for _ in range(rng.integers(2, 6)):
            S += u(0.8, 1.2) * burst(u(0, T_FRAMES - 6), u(1, 2.5)) * np.exp(-f / u(12, 20))
    elif cls == 8:
        t0 = u(0, 30)
        t1 = u(t0 + 60, T_FRAMES)
        S += line(u(30, 56) + u(0.3, 1.0) * np.sin(2 * np.pi * t / u(8, 16)), 0.9) * ((t >= t0) & (t < t1))
    else:
        S += burst(u(0, 60), u(15, 40)) * np.exp(-f / u(8, 15)) * (0.6 + 0.4 * rng.random((F_BINS, T_FRAMES)))
    S = u(0.5, 1.5) * S + u(0.05, 0.3) * rng.random((F_BINS, T_FRAMES)) * (1 + np.exp(-f / 30))
    return np.log1p(10 * S).astype(np.float32)


rng = np.random.default_rng(SEED)
X_spec = np.stack([make_spectrogram(c, rng) for c in range(len(SOUNDS)) for _ in range(N_SOUND)])
y_spec = np.repeat(np.arange(len(SOUNDS)), N_SOUND)
Xs_tmp, Xs_te, ys_tmp, ys_te = train_test_split(X_spec, y_spec, test_size=300, stratify=y_spec, random_state=SEED)
Xs_tr, Xs_va, ys_tr, ys_va = train_test_split(Xs_tmp, ys_tmp, test_size=300, stratify=ys_tmp, random_state=SEED)
mu, sd = Xs_tr.mean(), Xs_tr.std()
spec_t = lambda a: to_t((a - mu) / sd).unsqueeze(1)
Xs_tr_t, Xs_va_t, Xs_te_t = spec_t(Xs_tr), spec_t(Xs_va), spec_t(Xs_te)
ys_tr_t, ys_va_t = to_t(ys_tr, torch.long), to_t(ys_va, torch.long)
print(f"спектрограммы: train {Xs_tr.shape}, val {Xs_va.shape}, test {Xs_te.shape}")

In [ ]:
# --- 3.2 Audio Spectrogram Transformer ---
class AST(nn.Module):
    def __init__(self, n_classes=10, patch=8, d=128, depth=3, heads=4):
        super().__init__()
        self.patch_embed = nn.Conv2d(1, d, patch, patch)
        n_patches = (F_BINS // patch) * (T_FRAMES // patch)
        self.cls = nn.Parameter(torch.zeros(1, 1, d))
        self.pos = nn.Parameter(0.02 * torch.randn(1, n_patches + 1, d))
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d, dropout=0.1, activation='gelu', batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False)
        self.norm, self.head = nn.LayerNorm(d), nn.Linear(d, n_classes)

    def forward(self, x):
        x = self.patch_embed(x).flatten(2).transpose(1, 2)
        x = torch.cat([self.cls.expand(len(x), -1, -1), x], dim=1) + self.pos
        return self.head(self.norm(self.encoder(x)[:, 0]))


def spec_augment(x, max_f=8, max_t=16):
    b = len(x)
    f0 = torch.randint(0, F_BINS - max_f, (b, 1, 1, 1), device=x.device)
    fw = torch.randint(0, max_f + 1, (b, 1, 1, 1), device=x.device)
    t0 = torch.randint(0, T_FRAMES - max_t, (b, 1, 1, 1), device=x.device)
    tw = torch.randint(0, max_t + 1, (b, 1, 1, 1), device=x.device)
    fi = torch.arange(F_BINS, device=x.device).view(1, 1, -1, 1)
    ti = torch.arange(T_FRAMES, device=x.device).view(1, 1, 1, -1)
    mask = ((fi >= f0) & (fi < f0 + fw)) | ((ti >= t0) & (ti < t0 + tw))
    return x.masked_fill(mask, 0.0)


set_seed(SEED)
ast = AST().to(device)
print(f"AST: {count_params(ast):,} параметров, токенов: {ast.pos.shape[1]} (128 патчей + CLS)")
ast, ast_hist = train(ast, Xs_tr_t, ys_tr_t, Xs_va_t, ys_va_t, epochs=20, bs=32, lr=5e-4, augment=spec_augment)
ys_proba = predict_proba(ast, Xs_te_t)
ys_pred = ys_proba.argmax(1)
ast_acc = (ys_pred == ys_te).mean()
ast_f1 = f1_score(ys_te, ys_pred, average='macro')
print(f"время 20 эпох: {ast_hist['time']:.1f} c | best val acc {max(ast_hist['val_acc']):.3f} | "
      f"test acc {ast_acc:.3f} | test macro-F1 {ast_f1:.3f}")

In [ ]:
# --- 3.3 Визуализация: 10 тестовых спектрограмм, обучение, confusion matrix ---
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for c, ax in enumerate(axes.ravel()):
    i = np.where(ys_te == c)[0][0]
    ax.imshow(Xs_te[i], origin='lower', aspect='auto', cmap='magma')
    ok = ys_pred[i] == ys_te[i]
    ax.set_title(f"истина: {SOUNDS[ys_te[i]]}\nпрогноз: {SOUNDS[ys_pred[i]]} ({ys_proba[i, ys_pred[i]]:.0%})",
                 color='#0a7a0a' if ok else '#c62828', fontsize=10)
    ax.set_xlabel('время (кадр)')
    ax.set_ylabel('частота (бин)')
    ax.grid(False)
fig.suptitle('AST на тестовых спектрограммах (зеленый - верно, красный - ошибка)', y=1.0)
plt.tight_layout()
plt.show()

cm_s = confusion_matrix(ys_te, ys_pred)
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5), gridspec_kw={'width_ratios': [1, 1.2]})
ep_ast = np.arange(1, len(ast_hist['train_loss']) + 1)
axes[0].plot(ep_ast, ast_hist['train_loss'], color=COLORS['LSTM'], label='train loss')
ax2 = axes[0].twinx()
ax2.plot(ep_ast, ast_hist['val_acc'], color=COLORS['TCN'], label='val acc')
ax2.grid(False)
axes[0].set_xlabel('эпоха')
axes[0].set_ylabel('train loss', color=COLORS['LSTM'])
ax2.set_ylabel('val accuracy', color=COLORS['TCN'])
axes[0].set_title('Обучение AST')
axes[1].imshow(cm_s, cmap='Blues')
for i in range(len(SOUNDS)):
    for j in range(len(SOUNDS)):
        if cm_s[i, j]:
            axes[1].text(j, i, cm_s[i, j], ha='center', va='center', fontsize=8, color='white' if cm_s[i, j] > 15 else 'black')
axes[1].set_xticks(range(10), SOUNDS, rotation=45, ha='right')
axes[1].set_yticks(range(10), SOUNDS)
axes[1].set_xlabel('прогноз')
axes[1].set_ylabel('истина')
axes[1].set_title('Confusion matrix AST (test)')
axes[1].grid(False)
plt.tight_layout()
plt.show()

off_s = cm_s - np.diag(np.diag(cm_s))
top_conf = [(SOUNDS[i], SOUNDS[j], off_s[i, j]) for i, j in
            (divmod(int(k), 10) for k in np.argsort(off_s.ravel())[::-1][:5]) if off_s[i, j]]
print("Чаще всего путаются (истина -> прогноз):")
for a, b, n in top_conf:
    print(f"  {a:>14s} -> {b:<14s}: {n}")
report_s = classification_report(ys_te, ys_pred, target_names=SOUNDS, digits=3, output_dict=True)
print(classification_report(ys_te, ys_pred, target_names=SOUNDS, digits=3))

err_i = np.where(ys_pred != ys_te)[0][:6]
if len(err_i):
    fig, axes = plt.subplots(1, len(err_i), figsize=(3.2 * len(err_i), 3), squeeze=False)
    for ax, i in zip(axes[0], err_i):
        ax.imshow(Xs_te[i], origin='lower', aspect='auto', cmap='magma')
        ax.grid(False)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(f"{SOUNDS[ys_te[i]]} -> {SOUNDS[ys_pred[i]]}\n({ys_proba[i, ys_pred[i]]:.0%})", color='#c62828', fontsize=10)
    fig.suptitle('Ошибки AST (истина -> прогноз)', y=1.12)
    plt.show()

### Выводы по части 3

In [ ]:
recall = pd.Series({s: report_s[s]['recall'] for s in SOUNDS}).sort_values()
perfect = [s for s in SOUNDS if report_s[s]['recall'] == 1 and report_s[s]['precision'] == 1]
impulsive = {'тиканье часов', 'стук в дверь', 'птицы', 'гром', 'лай собаки'}

text = [f"- AST за {len(ast_hist['val_acc'])} эпох: test accuracy {ast_acc:.3f}, macro-F1 {ast_f1:.3f}, лучшая val accuracy "
        f"{max(ast_hist['val_acc']):.3f}, ошибок {int((ys_pred != ys_te).sum())} из {len(ys_te)}."]
if top_conf:
    text.append("- Чаще всего путаются: " + ', '.join(f"{a} -> {b} ({n})" for a, b, n in top_conf) + ".")
    n_imp = sum(a in impulsive and b in impulsive for a, b, _ in top_conf)
    if n_imp >= len(top_conf) / 2:
        text.append("- В основном это короткие импульсные звуки: такое событие занимает 1-2 патча по времени, "
                    "и его легко спутать с другим коротким событием.")
else:
    text.append("- Ошибок на тесте нет.")
if recall.iloc[0] < 1:
    text.append(f"- Хуже всего recall у классов {', '.join(f'{s} ({v:.2f})' for s, v in recall.head(3).items() if v < 1)}.")
if perfect:
    text.append(f"- Без ошибок распознаются: {', '.join(perfect)}.")
text.append("- AST работает со спектрограммой как с картинкой: патч 8x8 - небольшой участок по частоте и времени, "
            "attention связывает участки по всей спектрограмме, например гармоники на разных частотах в один момент "
            "(речь, лай) или повторы во времени (тиканье, сирена).")
text.append("- В отличие от обычных картинок ось частоты не инвариантна к сдвигу: низкий гул и высокий свист отличаются "
            "именно положением, поэтому позиционные эмбеддинги здесь обязательны. SpecAugment (зануление полос "
            "частот и интервалов времени) работает как регуляризация.")
text.append("- На реальных данных (ESC-50, Speech Commands) AST обычно инициализируют весами ViT с ImageNet или "
            "предобучают на AudioSet, без этого трансформеру нужно много данных.")
display(Markdown('\n'.join(text)))

In [ ]:
# ============================================================
# ИТОГОВЫЕ ВЫВОДЫ
# ============================================================
text = [
    f"1. ЭКГ: лучшая точность у {best} ({t.loc[best, 'Test acc']:.3f}), быстрее всего обучается {by_time.index[0]} "
    f"({by_time.iloc[0]:.0f} с), медленнее всего {by_time.index[-1]} ({by_time.iloc[-1]:.0f} с). "
    f"Меньше всего параметров у {smallest}.",
    f"2. Шум: самая устойчивая модель {rank.index[0]} (средняя accuracy при sigma > 0 {rank.iloc[0]:.3f}), "
    f"самая чувствительная {rank.index[-1]} ({rank.iloc[-1]:.3f}). Устойчивость зависит от того, как модель "
    "извлекает признаки: свертки и патчи обрабатывают сигнал окнами, рекуррентные модели читают его по точкам.",
    f"3. Аудио: AST на спектрограммах получил test accuracy {ast_acc:.3f}, ViT-подход переносится на аудио напрямую.",
    "4. Данные синтетические и проще реальных, сравнение сделано на одном seed, поэтому абсолютные значения "
    "accuracy выше, чем были бы на MIT-BIH или ESC-50.",
]
display(Markdown('\n'.join(text)))